# Stage 40 — Classificação com SVM

Objetivo: classificar as quatro palavras separadamente para cada participante. A padronização, seleção de características e SVM ficam no mesmo `Pipeline`.


## 1. Importações


In [352]:
from pathlib import Path
import pickle
import re

import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


from config.channel_mapping import INDICES_CHANNELS_MAPPING



## 2. Caminhos e parâmetros do experimento


In [353]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_02_feature_extraction_psd"
OUTPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_03_svm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#K_FEATURES = 3
N_SPLITS = 5
RANDOM_STATE = 42
CLASS_LABELS = [0, 1, 2, 3]


## 3. Descobrir os participantes


In [354]:
feature_files = sorted(INPUT_DIR.glob("sub-*_ses-*_features_psd.npy"))
subjects = sorted({
    re.match(r"(sub-\d+)", path.name).group(1)
    for path in feature_files
})
print("Participantes:", subjects)


Participantes: ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10']


## 4. Função para juntar as sessões

As três sessões de um participante são concatenadas no eixo das épocas.


In [355]:
def load_subject(subject):
    session_files = sorted(INPUT_DIR.glob(f"{subject}_ses-*_features_psd.npy"))
    session_features = []
    session_labels = []

    for feature_path in session_files:

        session_name = feature_path.name.replace("_features_psd.npy", "")
        labels_path = INPUT_DIR / f"{session_name}_labels.npy"

        session_features.append(np.load(feature_path))
        session_labels.append(np.load(labels_path))

    if not session_features:
        raise FileNotFoundError(f"Nenhuma sessão encontrada para {subject}")

    features = np.concatenate(session_features, axis=0)
    labels = np.concatenate(session_labels, axis=0)
    
    return features, labels


## 5. Inspecionar um participante


In [356]:
example_features, example_labels = load_subject(subjects[0])
print("Características:", example_features.shape)
print("Rótulos:", example_labels.shape)
print("Classes:", np.unique(example_labels, return_counts=True))


Características: (100, 107, 2)
Rótulos: (100,)
Classes: (array([0, 1, 2, 3]), array([25, 25, 25, 25]))


## 6. Construir o pipeline

Em cada fold, o scaler e o seletor são ajustados somente no conjunto de treino.


In [357]:
# model = Pipeline([
#     ("scaler", StandardScaler()),
#     ("selector", SelectKBest(score_func=f_classif, k=K_FEATURES)),
#     ("svm", SVC(kernel="rbf", C=1.0, gamma="scale")),
# ])

# cross_validation = StratifiedKFold(
#     n_splits=N_SPLITS,
#     shuffle=True,
#     random_state=RANDOM_STATE,
# )

# model


## 7. Função de avaliação de um participante


In [358]:
def evaluate_subject(subject, k_features):
    features, labels = load_subject(subject)
    
    n_channels = features.shape[1]
    n_measures = features.shape[2]

    # (épocas, canais, features) -> (épocas, canais * features)
    features = features.reshape(features.shape[0], -1)

    scores = []
    matrices = []
    selected_features_in_all_folds = []

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=k_features)),
        ("svm", SVC(kernel="rbf", C=1.0, gamma="scale")),
    ])

    cross_validation = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    ) 

    for train_index, test_index in cross_validation.split(features, labels):

        x_train = features[train_index]
        x_test = features[test_index]
        y_train = labels[train_index]
        y_test = labels[test_index]

        model.fit(x_train, y_train)

        y_predict = model.predict(x_test)

        scores.append(
            accuracy_score(y_test, y_predict)
        )

        matrices.append(
            confusion_matrix(
                y_test,
                y_predict,
                labels=CLASS_LABELS,
                normalize="true",
            )
        )

        indices = np.flatnonzero(
            model.named_steps["selector"].get_support()
        )

        feature_info = []

        for feature_idx in indices:

            channel_idx, measure_idx = np.unravel_index(
                feature_idx,
                (n_channels, n_measures)
            )

            feature_info.append({
                "feature_idx": int(feature_idx),
                "channel_index": int(channel_idx),
                "channel_name": INDICES_CHANNELS_MAPPING[channel_idx],
                "measure_index": int(measure_idx),
            })

        selected_features_in_all_folds.append(feature_info)

    scores = np.asarray(scores)
    matrices = np.asarray(matrices)

    return {
        "k_features": k_features,
        "scores": scores,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std(),
        "confusion_matrices": matrices,
        "mean_confusion_matrix": matrices.mean(axis=0),
        "selected_features_in_all_folds": selected_features_in_all_folds,
    }

## 8. Avaliar apenas um participante

Use esta célula para entender a saída antes de executar todos.


In [359]:
example_subject = subjects[0]
example_result = evaluate_subject(example_subject, 3)

print("Participante:", example_subject)
print("Acurácias dos folds:", example_result["scores"])
print("Média:", example_result["mean_accuracy"])
print("Matriz média:\n", example_result["mean_confusion_matrix"])


Participante: sub-01
Acurácias dos folds: [0.45 0.5  0.35 0.4  0.45]
Média: 0.43
Matriz média:
 [[0.36 0.08 0.28 0.28]
 [0.28 0.16 0.28 0.28]
 [0.24 0.12 0.48 0.16]
 [0.08 0.08 0.12 0.72]]


## 9. Avaliar e salvar todos os participantes


In [ ]:
n_channels = 128
n_measures = 2

for k_features in range(1, n_measures*n_channels+1): 
    for subject in subjects:
        result = evaluate_subject(subject, k_features)
        output_path = OUTPUT_DIR / f"k_features_{k_features}" / f"{subject}_svm_results_.pkl"
        
        # Cria a pasta e subpastas se não existirem
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with output_path.open("wb") as file:
            pickle.dump({subject: result}, file)
            
        print(subject, f"acurácia média={result['mean_accuracy']:.3f}")

    print("Stage 03 finalizado.")



sub-01 acurácia média=0.410
sub-02 acurácia média=0.292
sub-03 acurácia média=0.300
sub-04 acurácia média=0.358
sub-05 acurácia média=0.333
sub-06 acurácia média=0.214
sub-07 acurácia média=0.308
sub-08 acurácia média=0.300
sub-09 acurácia média=0.333
sub-10 acurácia média=0.300
Stage 03 finalizado.
sub-01 acurácia média=0.420
sub-02 acurácia média=0.450
sub-03 acurácia média=0.290
sub-04 acurácia média=0.367
sub-05 acurácia média=0.383
sub-06 acurácia média=0.223
sub-07 acurácia média=0.358
sub-08 acurácia média=0.340
sub-09 acurácia média=0.350
sub-10 acurácia média=0.308
Stage 03 finalizado.
sub-01 acurácia média=0.430
sub-02 acurácia média=0.433
sub-03 acurácia média=0.280
sub-04 acurácia média=0.400
sub-05 acurácia média=0.392
sub-06 acurácia média=0.251
sub-07 acurácia média=0.292
sub-08 acurácia média=0.380
sub-09 acurácia média=0.300
sub-10 acurácia média=0.325
Stage 03 finalizado.
sub-01 acurácia média=0.420
sub-02 acurácia média=0.433
sub-03 acurácia média=0.310
sub-04 acurác